# CIS I — Assignment 1: Pivot Calibration

This notebook walks through the pivot-calibration scenario from the
assignment (Figure 1): a pointer and a calibration object, each
carrying a marker body, observed by an optical tracker. Your job is to
**design** the marker-sphere layouts and workspace placements, then
**calibrate** the pointer's tip position `p_tip` to within about 2mm.

Everything runs against `src.simulation`, a self-contained simulator that plays the role of the physical tracker +
hand-guided robot + calibration sensor described in the handout. It injects measurement noise — your code never sees ground truth
except where explicitly noted for **validation/reporting** in Step 8 (real
hardware wouldn't give you that; the point there is to report your achieved
accuracy, not to use the ground truth in your calibration math itself).



In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
from src import simulation
from src import vct3, Rot, Frame

np.set_printoptions(precision=3, suppress=True)


In [2]:
# Global Parameters for the test setup

# Length of the pointer tip in millimeters
TIP_LENGTH_MM = 150  

# Work volume cube centered at 300 and extent 200 mm each side, based on Step 8
TEST_CUBE_CENTER = 300
TEST_CUBE_HALF_SIZE = 200

# Distance from the test cube center to the tracker position
TRACKER_DISTANCE = 1500

# x direction offset from the test cube center to the calibration object position mm
CALIB_OFFSET_XDIR = -400

# Keep all points within 50 mm of boundary, done as we had test points that accidentally crossed due to added error
VOLUME_SAFETY_MARGIN = 50

# This is the viewable volume by the camera, a trapezoidal 3D shape 
# Minimum distance/depth to tracker as well as maximum distance/depth
# Set to 2200 because 1000 + 1200 = 2200, as given in figure, so min + thickness
VOLUME_DISTANCE_MIN = 1000
VOLUME_DISTANCE_MAX = 2200

# trapezoid shorter base length is 1000 mm
VOLUME_HALF_WIDTH_AT_MIN = 500 
VOLUME_HALF_WIDTH_AT_MAX = 1000

# Maximum pointer orientation rel to vertical direction
MAX_TILT_IN_RAD = np.deg2rad(45.0)     

# Found from lit. that adding basic increments can lead to repeats in overall rotation
# Used an irrational angle known as golden angle to avoid this and try different pointer rotations despite crossing 360 deg.
# https://www.mdpi.com/2226-4310/12/11/1006?; added after having high error in results to improve calibration
# Also have an irrational axial centerline increment with this to make sure orientation tested changes
IRRATIONAL_ANGLE_INCREMENT = np.pi * (3.0 - np.sqrt(5.0))  
SPIN_INCREMENT = np.sqrt(2.0)

# Number of unknowns in the pivot calibration problem
PIVOT_UNKNOWNS = 6

# half-extent of the near body in x and y, same for z in line below
NEAR_BODY_FROM_CENTER_XY = 300   
NEAR_BODY_FROM_CENTER_Z = 300

# minimum gap between any sphere and the test cube
NEAR_BODY_MINCLEARANCE = 50

# assignment spec max error
SPEC_MAX_ERROR = 2.0

# maximum tilt for same cone shape
SPEC_MAX_TILT_DEG = np.rad2deg(MAX_TILT_IN_RAD)

# Number of Step 8 test poses sampled from the full 42,875 pose grid
NUM_TEST_POSES = 100

In [3]:
# Helper Methods

def check_within_tracker_volume(F_tracker, points_ws, margin = VOLUME_SAFETY_MARGIN):
    """
    Header: check which workspace points are within the tracker's set viewing area with a given margin.

    Input:
        F_tracker: Frame, tracker pose relative to the workspace.
        points_ws: List[vct3], points in workspace coordinates.
        margin: float, required margin from boundary

    Returns:
        An np.ndarray of shape (len(points_ws),).
        True means the point is safely inside the viewing volume, false if not.
    """

    # Convert workspace points to tracker coordinates
    # Since F_tracker converts to workspace coordinates, need inverse (in frame class, given)
    tracker_points = [F_tracker.inv() * p for p in points_ws]

    # Get x, y, and depth coordinates
    x = np.array([p.vec[0] for p in tracker_points])
    y = np.array([p.vec[1] for p in tracker_points])
    depth = np.array([p.vec[2] for p in tracker_points])

    # Relative fraction of opening over 1200 mm from front view, used to calculate width at a given depth
    opening_frac = (depth - VOLUME_DISTANCE_MIN)/ (VOLUME_DISTANCE_MAX - VOLUME_DISTANCE_MIN)
    half_width = VOLUME_HALF_WIDTH_AT_MIN + opening_frac * (VOLUME_HALF_WIDTH_AT_MAX - VOLUME_HALF_WIDTH_AT_MIN)

    # Check whether each point is safely inside the volume, i.e. within margin
    inside = (
        (depth >= VOLUME_DISTANCE_MIN + margin)
        & (depth <= VOLUME_DISTANCE_MAX - margin)
        & (np.abs(x) <= half_width - margin)
        & (np.abs(y) <= half_width - margin)
    )

    return inside


def require_inside_tracker_volume(label, F_tracker, points_ws):
    """
    Main function: Raises ValueError if any point is outside the tracker viewing volume.

    input:  label, a str for point label for specifying which points outside viewing volume
            F_tracker: Frame, tracker pose relative to the workspace.
            points_ws: List[vct3], points in workspace coordinates.
    output: None
    """

    # array of booleans, for each point if true or false
    inside = check_within_tracker_volume(F_tracker, points_ws)
    # total number of points that are false as stored in inside vector
    outside = (~inside).sum()

    # if any points outside are detected, give prop. of points outside viewing volume
    if outside > 0:
      raise ValueError (label + ": " + str(outside) + " of " + str(len(points_ws))
          + " points are outside the tracker viewing volume"
      )

## Step 1: Design the pointer's marker body

Choose approximate local marker-sphere positions `a_ptr,k` (a `List[vct3]`,
relative to the pointer's own coordinate frame) and the pointer tip's
approximate local position `p_tip_approx`. These are your
*nominal design values* — the simulator will independently perturb the
*actual* manufactured positions by a small, noise amount (bounded at
2mm).


In [4]:
# 5 Pointer local positions that are non-coplanar and asymmetric
# Spread out several cm apart to avoid overlap
ptr_local_positions = [
    vct3(0, 0, 0),
    vct3(110, 0, 10),
    vct3(-40, 70, 10),
    vct3(-35, -60, 30),
    vct3(50, 50, 60),
]

# Tip is below the balls in -z direction. Far enough for a realistic reach,
# Short enough to avoid error at the tip.
ptr_tip_approx = vct3(0, 0, -TIP_LENGTH_MM)


## Step 1.1: Design the calibration object's marker body

Same idea, for the calibration object's local marker positions `a_cal,k`.


In [5]:
# 6 Calibration local positions that are non-coplanar and asymmetric
# Spread out several cm apart to avoid overlap
cal_local_positions = [
    vct3(-110, -90, -60),
    vct3(125, -95, -30),
    vct3(130, 95, -80),
    vct3(-120, 90, -45),
    vct3(0, -130, -100),
    vct3(15, 135, -20),
]


## Step 2: Approximate workspace poses

Specify approximate poses (as `Frame` class from src) for the tracking camera
(`F_tracker`) and the calibration object (`F_cal`), relative to the
workspace. The tracker's usable volume is the tetrahedral region in the
handout's Figure 2 (roughly 100-200cm in front of the tracker, widening
from ~120cm to ~200cm across) — place the calibration object, and later
the pointer, so they're inside that volume.


In [6]:

# Camera tracker flip 180 degrees to look straight down to test cube
# Center TRACKER_DISTANCE mm above the test cube 
F_tracker_approx = Frame(Rot('x', np.pi), vct3(TEST_CUBE_CENTER, TEST_CUBE_CENTER, TEST_CUBE_CENTER + TRACKER_DISTANCE))

# Calibration object pose no tilt relative to the test cube,
# Offset sideways in the x direction by CALIB_OFFSET_XDIR, 
# level with the test cube in the y and z directions
F_cal_approx = Frame(Rot.eye(), vct3(TEST_CUBE_CENTER + CALIB_OFFSET_XDIR, TEST_CUBE_CENTER, TEST_CUBE_CENTER))

# Get the 8 corners from the test cube center, using the half side lengths in each direction
cube_corners = []
for sx in (-1, 1):
    for sy in (-1, 1):
        for sz in (-1, 1):
            corner = vct3(
                TEST_CUBE_CENTER + sx * TEST_CUBE_HALF_SIZE,
                TEST_CUBE_CENTER + sy * TEST_CUBE_HALF_SIZE,
                TEST_CUBE_CENTER + sz * TEST_CUBE_HALF_SIZE
            )
            cube_corners.append(corner)

# Check that all cube corners are inside the camera tracker's viewable volume
# Raises ValueError if any point is outside the tracker viewing volume
require_inside_tracker_volume("test cube corners, step 2", F_tracker_approx, cube_corners)

# Convert each calibration ball from the object's own frame into room (workspace)
# coordinates, so we can check whether the tracker can actually see them
calibration_markers = [F_cal_approx * marker for marker in cal_local_positions]

# Check that all the calibration markers, now in workspace ref frame, are in the cameratracker's viewable volume
# Raises ValueError if any point is outside the tracker viewing volume
require_inside_tracker_volume("calibration markers",F_tracker_approx, calibration_markers)


## Step 3: Build the simulation

Fully provided — this just wires up your Step 1/2 design choices through
`simulation`'s `DefineMarkerBody` / `PlaceMarkerBody` / `CreateTracker`
equivalents. `ptr_placed`/`cal_placed` are `PlacedBody` handles: each pairs
a `MarkerBody` with its hidden actual placement, and gets passed to
`simulation.sample_marker_bodies(...)` whenever you want a reading.


In [7]:
ptr_mb = simulation.define_marker_body("ptr_mb", ptr_local_positions)
cal_mb = simulation.define_marker_body("cal_mb", cal_local_positions)

cal_placed = simulation.place_marker_body(cal_mb, F_cal_approx)
# The pointer's own body doesn't have a meaningful "workspace placement" yet
# It moves every time it's hand-guided (Step 4-6). 
# precision="precise" matches a robot-controlled pose rather than a coarsely, manually placed
# static object (see place_marker_body's docstring).
ptr_placed = simulation.place_marker_body(ptr_mb, Frame.eye(), precision="precise")

ptr = simulation.define_pointer("ptr1", ptr_tip_approx, ptr_mb)
trk = simulation.create_tracker("trk1", F_tracker_approx)

print("simulation built:", ptr_mb.name, cal_mb.name, ptr.name, trk.name)


simulation built: ptr_mb cal_mb ptr1 trk1


## Step 3.1: Refine `a_ptr,k` / `a_cal,k`

> "Observations of marker positions can be used to determine accurate
> estimates of relative positions a_ptr,k of markers on pointer and
> calibration objects... **Hint**: You probably want to take multiple
> readings in order to reduce the uncertainty associated with the a_k
> values."


In [8]:
from data_types.uncertain_types import uvct3

def refine_marker_body(trk, placed, number_readings=30, rounds=3):
    """
    Sharpen a marker body's local ball positions by averaging many tracker readings and computing pose estimate.

    input:
        trk - Tracker (camera) to read with
        placed - PlacedBody whose body's nominal marker positions are refined in place
        number_readings - int, how many readings/snapshots averaged per round
        rounds - int, number of refinement rounds
    output: MarkerBody, the same body object, with its nominal_marker_positions
            overwritten by the refined estimates (updated in place)
    """
     
    # the body we are refining (pointer or calibration object)
    body = placed.body

    # Repeat the refinement process multiple (rounds) times
    for _ in range(rounds):
        # one snapshot's worth of readings ball positions
        all_readings = []

        # Take number_readings snapshots so the random fuzz is averaged out
        for _ in range(number_readings):
            # One camera snapshot of just this body
            obs = simulation.sample_marker_bodies(trk, [placed])[body.name]

            # Camera's estimate of this body's pose (R, p), in the tracker frame
            R = obs.frame.R.matrix           
            p = obs.frame.p.vec.ravel()       

            # The noisy ball readings, in the tracker's frame
            B = np.array([b.p.vec.ravel() for b in obs.markers])

            # Re-express balls in the body's own frame: a = R^T (b - p)
            A_local = (B - p) @ R
            all_readings.append(A_local)

        # Average over snapshots per ball
        average = np.mean(np.array(all_readings), axis=0)

        # Save the average and saves it back into the body
        body.set_marker_positions([vct3(row) for row in average])

    return body

# The rigid transform that takes the body's true marker positions onto its refined ones.
def true_to_refined_frame(trk, placed):
    """
    Best rigid transform taking a body's true marker positions onto its refined ones.

    input:  trk - Tracker, used for its procrustes_solver 
            placed - PlacedBody whose refined geometry is compared with ground truth
    output: Frame, maps coordinates in the body's true frame into its refined frame
    """
    # Hidden true ball positions vs our refined estimates, both in body local coordinates
    actual = [m.p for m in placed.body.get_actual_marker_positions()]
    estimated = [uvct3(p) for p in placed.body.nominal_marker_positions]
     # Best fit rotation + translation between the two point sets
    return trk.procrustes_solver(actual, estimated)

def marker_misfit_rms(trk, placed):
    """
    Measure how close the refined ball positions are to the true ones (for reporting only, not calibration).

    input:
        trk - Tracker (camera), used for its best fit alignment method
        placed - PlacedBody whose refined marker positions are checked against ground truth
    output: float, RMS ball position error in mm
    """
    # actual ball positions and our refined estimates, both in body local coordinates
    T = true_to_refined_frame(trk, placed)

    # Distance between each aligned true ball and its refined estimate
    errors = [(T * a - r).norm()
              for a, r in zip([m.p for m in placed.body.get_actual_marker_positions()], placed.body.nominal_marker_positions)]

    # Root mean square over all balls
    return float(np.sqrt(np.mean(np.square(errors))))


# Refine both bodies (pointer and calibration object) average many readings to sharpen their ball positions
refine_marker_body(trk, ptr_placed)
refine_marker_body(trk, cal_placed)

# Pointer: show refined positions and the error vs the true geometry (mm)
print("pointer refined:", ptr_placed.body.nominal_marker_positions)
print("pointer misfit RMS:", marker_misfit_rms(trk, ptr_placed))

# Calibration object: same, refined positions and error vs truth (mm)
print("calibration refined:", cal_placed.body.nominal_marker_positions)
print("calibration misfit RMS:", marker_misfit_rms(trk, cal_placed))

pointer refined: [vct3(x=-0.71, y=-0.25, z=0.96)
Internal Vector:
[[-0.714]
 [-0.252]
 [ 0.958]], vct3(x=110.60, y=0.89, z=9.63)
Internal Vector:
[[110.602]
 [  0.894]
 [  9.63 ]], vct3(x=-39.94, y=69.47, z=9.16)
Internal Vector:
[[-39.94 ]
 [ 69.468]
 [  9.159]], vct3(x=-35.53, y=-59.42, z=29.76)
Internal Vector:
[[-35.531]
 [-59.424]
 [ 29.764]], vct3(x=50.58, y=49.31, z=60.49)
Internal Vector:
[[50.583]
 [49.315]
 [60.489]]]
pointer misfit RMS: 0.004754634552612491
calibration refined: [vct3(x=-110.01, y=-90.30, z=-61.04)
Internal Vector:
[[-110.009]
 [ -90.302]
 [ -61.038]], vct3(x=124.76, y=-94.62, z=-29.86)
Internal Vector:
[[124.755]
 [-94.617]
 [-29.86 ]], vct3(x=130.81, y=95.13, z=-80.34)
Internal Vector:
[[130.811]
 [ 95.129]
 [-80.344]], vct3(x=-120.18, y=89.22, z=-44.35)
Internal Vector:
[[-120.183]
 [  89.222]
 [ -44.353]], vct3(x=-0.60, y=-129.55, z=-99.24)
Internal Vector:
[[  -0.6  ]
 [-129.549]
 [ -99.237]], vct3(x=15.23, y=135.12, z=-20.17)
Internal Vector:
[[ 15.226]

## Steps 4-6: Collect pivot-calibration observations

For each observation `j`: hand-guide the pointer near the calibration
object at some desired orientation `R_desired,j`, take one simultaneous
tracker reading of *both* bodies, and read the calibration sensor. Loop
skeleton is provided; you choose how many observations to take and how to
vary `R_desired,j`


In [9]:
N_OBSERVATIONS = 50 # Balance number of observations for calibration

def desired_orientation_incrementation(j, n_observations):
    """
    Desired pointer orientation relative to the calibration object for a set observation index j

    input:  j, is the current index, i.e. what observation we are currently on
            n_observations, an integer, number of observations being taken
    output: 
            Rot, a rotation matrix formed as product of rotation about z-axis, rotation due to tilt, and spin,
                 with tilt maxed out at 45 degrees, and tilt is pointer "tilt" relative to vertical
    Note: this segment of code was AI assisted for the formulas and for identifying whether the matrices had compatible dimensions for mutliplication. 
    Specific prompt listed in writeup.
    """
    # We got these formulas from https://isgwww.cs.uni-magdeburg.de/graphics/teaching/2017/ais/materials/stanford_math_for_cg.pdf?
    # and https://mathworld.wolfram.com/GoldenAngle.html as a way to make sure uniformly sampling spherical space with tip
    cos_tilt = 1.0 - (j + 0.5) / n_observations * (1.0 - np.cos(MAX_TILT_IN_RAD)) 
    
    # use irrational angle increments for rotating about z-axis, cited when defined above
    azimuth = j * IRRATIONAL_ANGLE_INCREMENT
    # Converting the incremented value to between 0 and 2 pi radians to get spin angle
    spin = 2.0 * np.pi * ((j * SPIN_INCREMENT) % 1.0)
    # Rotational matrices all multiply to get final rotation after increment
    return Rot('z', azimuth) * Rot('y', np.arccos(cos_tilt)) * Rot('z', spin)

# Per observation results: pointer pose, calibration object pose (both seen by the camera tracker),
# and the calibration sensor's tip reading (None if the tip landed outside the 5 mm sensor range)
F_ptr_list = []
F_cal_list = []
sensed_tip_list = []
for j in range(N_OBSERVATIONS):
    # The desired pointer tilt for this pose, evenly spread over the cone via the spiral method
    R_desired = desired_orientation_incrementation(j, N_OBSERVATIONS)
    # The hand guide for the pointer
    hg = simulation.hand_guide_pointer(ptr, ptr_placed, cal_placed, R_desired)
    # one snapshot of both bodies, so the tracker's wobble is shared and cancels later
    obs = simulation.sample_marker_bodies(trk, [ptr_placed, cal_placed])
    # The calibration sensor's tip reading
    sensed = simulation.sense_tip(ptr, hg)

    # Store this observation
    F_ptr_list.append(obs["ptr_mb"].frame)
    F_cal_list.append(obs["cal_mb"].frame)
    sensed_tip_list.append(sensed)

print(f"collected {len(F_ptr_list)} observations")


collected 50 observations


`sensed_tip_list` holds each observation's high-accuracy calibration-sensor
reading (or `None` if that observation ended up outside the sensor's 5mm
range) — useful as an independent sanity check on your calibration later,
since it measures the tip position a completely different way (directly,
rather than through the pointer's marker geometry).


## Step 7: Compute a calibrated value for `p_tip`

This is the assignment's core deliverable — implement it yourself and specify the details of implementation in the report.

**Corner case to consider**: What are the corner cases you want to analyse here?

**Minimum data**: how many observations, and what about their rotations,
are needed for the stacked system to actually be solvable? How do you define whether a system is solvable here? Include that in your report


In [10]:
def relative_poses(F_ptr_list, F_cal_list):
    """
    Calculates pointer pose in the calibration frame for each observation, which is F_j = (F_cal,j)^-1 * F_ptr,j

    input:  F_ptr_list, F_cal_list, each of which are list of frames for each pointer frame and the calibration object's frame
    output is a List[Frame]
    """
    relative_frames = []

    # For each observation
    for i in range(len(F_ptr_list)):
        # where the camera saw the pointer
        pointer_frame = F_ptr_list[i]
        # where the camera saw the calibration object
        calibration_frame = F_cal_list[i]

        # Express the pointer frame relative to the calibration frame using the same equation as in header
        relative_frame = calibration_frame.inv() * pointer_frame

        relative_frames.append(relative_frame)

    return relative_frames

def build_pivot_system(relative_frames, sensed_tips):
    """
    Stack the equations per observation [Rj -I]x = bj into Ax=b, meaning stack all observations together

    input:  relative_frames, a List[Frame] where each F_j = (R_j, p_j)
            sensed_tips, a List[vct3], s_j for each frame 
    output: (A, b) with A of shape (3n, 6) and b of shape (3n,)
    """
    A_rows = []
    b_values = []

    # for all observations
    for i in range(len(relative_frames)):

        frame = relative_frames[i]
        sensed_tip = sensed_tips[i]

        # Rotation matrix for this observation
        rotation_matrix = frame.R.matrix

        # [R -I] for this observation
        A_row = np.hstack([rotation_matrix,-np.eye(3)])
        # combine into one matrix
        A_rows.append(A_row)

        # right side is deviation of tip, now stored in 1D
        b_row = sensed_tip - frame.p
        b_row = b_row.vec.ravel()

        b_values.append(b_row)

    # Stack all of the 3x6 matrices
    A = np.vstack(A_rows)

    # Combine all b vectors into one long vector
    b = np.concatenate(b_values)

    return A, b


def fit_pivot(F_ptr_list, F_cal_list, sensed_tip_list=None):
    """
    Least-squares pivot calibration

    input:
        F_ptr_list is a List(Frame), pointer marker body pose in the tracker frame, one per observation.
        F_cal_list is a List(Frame), calibration object pose in the tracker frame, one per observation.
        sensed_tip_list - is a list of vectors, may or may not be present if not sensed
    output:
        p_tip: estimated pointer tip position in the pointer frame mm
        offset: estimated pivot position, so an offset to p_tip mm
        covariance, a 3x3 covariance matrix for the tip position 
        rms_error, RMS residual error mm
        num_usable - number of observations used in the calibration
        ValueError if not 3 observations of sensor tip present or if not enough variation in rotations for proper calibration
    """
    # account for when no sensor tips detected 
    if sensed_tip_list is not None:
        sensed = sensed_tip_list
    else:
        sensed = [vct3()] * len(F_ptr_list)

    # need at least three observations
    usable = [j for j, s in enumerate(sensed) if s is not None]
    if len(usable) < 3:
        raise ValueError(f"fit_pivot: need at least 3 usable observations, got {len(usable)}")

    # compute pointer position relative to the calibration object across all observations
    ptr = [F_ptr_list[j] for j in usable]
    cal = [F_cal_list[j] for j in usable]
    tips = [sensed[j] for j in usable]

    frames = relative_poses(ptr, cal)
    # Stack system across observations
    A, b = build_pivot_system(frames, tips)

    # If matrix is singular, not enough rotation directs
    # Pointer movements weren't varied enough for the calibration math to figure out the tip location reliably.
    if np.linalg.matrix_rank(A) < 6:
        raise ValueError("Rotations are degenerate.")

    # find least squares estimate of x that minimizes sum of squared error
    x, residuals, rank, singular_vals= np.linalg.lstsq(A, b, rcond=None)
    # residual given best estimate
    residual = A @ x - b
    # error variance formula (r^T * r) / (m - n)
    sigma_squared = float(residual @ residual) / (A.shape[0] - PIVOT_UNKNOWNS)
    # Least Squares covariance formula
    covariance = sigma_squared * np.linalg.inv(A.T @ A)

    # store and return values and error
    p_tip = vct3(x[:3])
    offset = vct3(x[3:])
    p_tip_covariance = covariance[:3, :3]
    # compute rms of error with the number of usable vectors (not NONE)
    rms_error = float(np.sqrt(np.mean(residual ** 2)))
    num_usable = len(usable)

    return p_tip, offset, p_tip_covariance, rms_error, num_usable


def compute_pivot_calibration(F_ptr_list, F_cal_list, sensed_tip_list=None):
    """
    Calibrated tip position of the pointer relative to the pointer frame

    input:
        F_ptr_list - List[Frame], the pointer marker body's pose relative to
                the tracker for each observation j (Fptr,j)
        F_cal_list - List[Frame], the calibration object's pose relative to
                the tracker for each observation j (Fcal,j), same order/length
    output:
        p_tip - vct3, the calibrated tip position in the pointer's local frame
            cov_estimate - np.ndarray (3x3), an estimated covariance for p_tip
                (e.g. from the least-squares fit's residuals)
        """
    # Obtain the values returned from pivot fitting and return the position of the tip and covariance, which are the only needed outputs
    p_tip, offset, covariance, rms_error, num_usable = fit_pivot(F_ptr_list, F_cal_list, sensed_tip_list)

    return p_tip, covariance

p_tip_est, p_tip_cov = compute_pivot_calibration(F_ptr_list, F_cal_list, sensed_tip_list)

In [11]:
print("calibrated p_tip:", p_tip_est)
print("estimated covariance (mm^2):\n", p_tip_cov)
print("estimated std-dev per axis (mm):", np.sqrt(np.diag(p_tip_cov)))


calibrated p_tip: vct3(x=-0.03, y=0.18, z=-150.17)
Internal Vector:
[[  -0.033]
 [   0.176]
 [-150.171]]
estimated covariance (mm^2):
 [[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
estimated std-dev per axis (mm): [0.004 0.004 0.008]


## Step 8: Validate against test poses

> "**Hint**: You may want to consider placing additional marker bodies
> close to (but not inside) the cube defined by the test volume. However,
> this will also depend on your design for and placement of the
> calibration object."

The nominal test-pose grid is `theta_x, theta_y, theta_z` each in
`{0, +-pi/6, +-pi/3, +-pi/2}` and `p_x, p_y, p_z` each in
`{300, 300+-100, 300+-200}` mm. For each test pose, the assignment requires a
**single observation only** (no averaging).

Think about why the hint suggests an *additional*, nearby reference body?

In [12]:
# Rigid rectangular prism with a sphere at each of the 8 corners
# Its spheres sit outside the test cube
near_local_positions = [
    vct3(sx * NEAR_BODY_FROM_CENTER_XY, sy * NEAR_BODY_FROM_CENTER_XY, sz * NEAR_BODY_FROM_CENTER_Z)
    for sx in (-1, 1) for sy in (-1, 1) for sz in (-1, 1)
]
# Frame at the test cube center, no rotation
F_near_approx = Frame(Rot.eye(), vct3(TEST_CUBE_CENTER, TEST_CUBE_CENTER, TEST_CUBE_CENTER))

# Move the reference-body spheres into workspace coordinates
near_spheres_ws = [F_near_approx * point for point in near_local_positions]

# Clearance of each sphere from the test cube
clearances = []
for point in near_spheres_ws:
    # vector from cube center to sphere
    offset = point.vec.ravel() - TEST_CUBE_CENTER
    # distance from cube center to sphere
    distance_from_center = np.max(np.abs(offset))
    clearances.append(distance_from_center - TEST_CUBE_HALF_SIZE)

# Smallest gap between any sphere and the cube surface
cube_clearance = min(clearances)

# Keep every sphere a minimum distance away from the cube
if cube_clearance < NEAR_BODY_MINCLEARANCE:
    raise ValueError(f"Reference spheres are only {cube_clearance:.1f} mm from the cube")

# Make sure all spheres are inside the tracker's viewable volume
require_inside_tracker_volume("reference-body spheres", F_tracker_approx, near_spheres_ws)

# Build the reference body, place it (coarse, manual placement), and refine it like Step 3.1
near_mb = simulation.define_marker_body("near_mb", near_local_positions)
near_placed = simulation.place_marker_body(near_mb, F_near_approx)
refine_marker_body(trk, near_placed)

Refine this new body's geometry the same way you refined the pointer's and
calibration object's (Step 3b) before using it.


Now the single-observation test loop. For each nominal test pose `F_t`:
place the pointer there (small, robot-precision placement error, same as
the real system), take one simultaneous reading of the pointer and your
Step-8 reference body, and predict the tip's position (relative to that
reference body) using your calibrated `p_tip`.

**Ground truth for reporting only**: `PlacedBody.actual_placement` and
`MarkerBody.actual_marker_positions` hold the hidden values the simulator
sampled — you have access to them here only because this notebook doesn't
gate them behind a separate eval mode (per the handout: "During debugging
you will have access to all of the ... values. These will not be available
during evaluation."). Use them *only* to compute and report your achieved
error in this section — using them anywhere in your actual calibration math
(Steps 3b/7) kills the point of the assignment.


In [13]:
# thetas and offsets listed in step 8
thetas = [0, np.pi / 6, -np.pi / 6, np.pi / 3, -np.pi / 3, np.pi / 2, -np.pi / 2]
offsets = [0, 100, -100, 200, -200]


def sample_test_poses(n_poses, seed = 42):
    """
    A uniform random subset of test poses, with no repeats of the full grid.

    input:  n_poses is int, num of poses to sample
    output: List[Frame], nominal test poses Frame(Rot.xyz(rx, ry, rz), (300+tx, 300+ty, 300+tz))
    """
    # Grid incorporating all offsets/thetas in each direction, thousnds of possible poses
    rng = np.random.default_rng(seed)
    grid_shape = (len(thetas),) * 3 + (len(offsets),) * 3
    # sample number of poses from thousands possible
    chosen = rng.choice(int(np.prod(grid_shape)), size=n_poses, replace=False)
    poses = []
    for number in chosen:
        # Obtain index in grid from which the sample taken
        indices = np.unravel_index(number, grid_shape)

        # obtain offset and rotational values for each direction
        rx_index = indices[0]
        ry_index = indices[1]
        rz_index = indices[2]

        tx_index = indices[3]
        ty_index = indices[4]
        tz_index = indices[5]

        rx = thetas[rx_index]
        ry = thetas[ry_index]
        rz = thetas[rz_index]

        tx = offsets[tx_index]
        ty = offsets[ty_index]
        tz = offsets[tz_index]

        # Rotation matrix in all directions
        rotation = Rot.xyz(rx, ry, rz)
        # Displacement portion of frame
        position = vct3(
            TEST_CUBE_CENTER + tx,     
            TEST_CUBE_CENTER + ty,
            TEST_CUBE_CENTER + tz
        )

        pose = Frame(rotation, position)
        poses.append(pose)
    return poses


def pointer_points_ws(F_t):
    """
    Marker positions and tip of the pointer when placed at F_t, in workspace coordinates.
    
    Input: F_t, pose of the pointer marker in current setting
    Output: returns all pointer markers in workspace coordinates
    """
    return [F_t * p for p in ptr_mb.nominal_marker_positions] + [F_t * ptr_tip_approx]


def pointer_tilt_deg(F_t):
    """
    Angle between the pointer's long axis (its local z) and vertical, in degrees from 0-90.
    
    input:  F_t, Frame, pose of the pointer in the workspace
    output: float, tilt from vertical in degrees
    """
    cos_tilt = np.clip(F_t.R.matrix[2, 2], -1.0, 1.0)
    return float(np.rad2deg(np.arccos(cos_tilt)))

def evaluate_test_pose(F_t_nominal, near_gauge):
    """
    One test pose, one observation: predicted tip position (relative to the reference body) vs ground truth.

    input:  F_t_nominal - Frame, nominal pointer pose in the workspace
    output: (error_mm, gauge_aligned_error_mm) - distance between predicted and true tip position,
            the second after expressing the truth in the reference body's refined coordinates
    Note: this segment of code was AI assisted for explaining gauge vs non-gauge results. Noted in handout.
    """
    # place marker bodies using precise feature
    test_placed = simulation.place_marker_body(ptr_mb, F_t_nominal, precision="precise")
    # collect simulated frames between test and near
    obs = simulation.sample_marker_bodies(trk, [test_placed, near_placed])
    # Convert point to tracker coordinates and then express it relative to nearby reference body (work volume)
    predicted = obs["near_mb"].frame.inv() * (obs["ptr_mb"].frame * p_tip_est)

    # Ground truth for reporting the squared error etc.
    true_tip = near_placed.actual_placement.inv() * (test_placed.actual_placement * ptr.get_actual_tip_position().p)
    # Calculate error in prediction and actual answer, as well as including error incorporating the natural offset
    return (predicted - true_tip).norm(), (predicted - near_gauge * true_tip).norm()


# run set number of tests
near_gauge = true_to_refined_frame(trk, near_placed)
test_poses = sample_test_poses(NUM_TEST_POSES)
tilts_sampled = np.array([pointer_tilt_deg(F_t) for F_t in test_poses])
n_in_cone = np.sum(tilts_sampled <= SPEC_MAX_TILT_DEG)
print(f"Test poses within the tilt limit ({SPEC_MAX_TILT_DEG:.0f}°): {n_in_cone}/{NUM_TEST_POSES}")

# ensure all pointer markers are viewable by tracker
require_inside_tracker_volume("pointer markers/tip over the test poses", F_tracker_approx,
                              [p for F_t in test_poses for p in pointer_points_ws(F_t)])
# check ground truth and compute error
near_gauge = true_to_refined_frame(trk, near_placed)

# collect errors for both calibration only and calibration including offset
test_results = []
gauge_aligned_errors = []
for F_t_nominal in test_poses:
    error_mm, aligned_mm = evaluate_test_pose(F_t_nominal, near_gauge)
    test_results.append((F_t_nominal, error_mm))
    gauge_aligned_errors.append(aligned_mm)

Test poses within the tilt limit (45°): 12/100


## Achieved performance

Summarize the tip-position error across the test-pose grid against the
handout's 2mm spec.


In [14]:
def summarize_errors(errors_mm):
    """
    Store the error and compute metrics on it

    input:  sequence of tip-position errors in mm
    output: dict with n, mean, rms, p95, max, and the percentage within spec error limit
    """
    e = np.asarray(errors_mm, dtype=float)
    # No poses in this group, return NaNs instead of crashing
    if e.size == 0:
        return dict(n=0, mean=np.nan, rms=np.nan, p95=np.nan, max=np.nan, within=np.nan)
    # count, mean, RMS, 95th percentile, max, and % of poses within the 2 mm spec
    return dict(n=len(e), mean=e.mean(), rms=np.sqrt(np.mean(e ** 2)), p95=np.percentile(e, 95),
                max=e.max(), within=100.0 * np.mean(e <= SPEC_MAX_ERROR))


# This code below was primarily AI assisted for formatting of a table with error, results, etc done manually but formatted via AI
# Pull the error arrays and each pose's tilt out of the Step 8 results
raw_errors = np.array([e for _, e in test_results])
aligned_errors = np.array(gauge_aligned_errors)
tilts = np.array([pointer_tilt_deg(F_t) for F_t, _ in test_results])
# True for poses inside the handout's 45-degree tilt spec
within_spec_cone = tilts <= SPEC_MAX_TILT_DEG

# Table header, then one block per pose group (all / in-spec tilt / beyond-spec tilt)
print(f"{'poses':<28}{'error type':<14}{'n':>5}{'mean':>8}{'rms':>8}{'95%':>8}{'max':>8}{'<=2mm':>8}")
for label, mask in (("all poses", np.ones_like(tilts, dtype=bool)),
                    (f"tilt <= {SPEC_MAX_TILT_DEG:.0f} deg (spec)", within_spec_cone),
                    (f"tilt >  {SPEC_MAX_TILT_DEG:.0f} deg", ~within_spec_cone)):
    # Aligned error (headline metric)
    s = summarize_errors(aligned_errors[mask])
    print(f"{label:<28}{'aligned':<14}{s['n']:>5}{s['mean']:>8.3f}{s['rms']:>8.3f}{s['p95']:>8.3f}{s['max']:>8.3f}{s['within']:>7.1f}%")
    # Raw error (for comparison)
    s = summarize_errors(raw_errors[mask])
    print(f"{'':<28}{'raw':<14}{s['n']:>5}{s['mean']:>8.3f}{s['rms']:>8.3f}{s['p95']:>8.3f}{s['max']:>8.3f}{s['within']:>7.1f}%")

# Report the single worst pose
worst = int(np.argmax(aligned_errors))
print(f"\nworst pose (aligned): tilt {tilts[worst]:.0f} deg, error {aligned_errors[worst]:.3f} mm "
      f"at nominal position {test_results[worst][0].p.vec.ravel()}")

poses                       error type        n    mean     rms     95%     max   <=2mm
all poses                   aligned         100   0.044   0.047   0.074   0.121  100.0%
                            raw             100   0.336   0.363   0.586   0.658  100.0%
tilt <= 45 deg (spec)       aligned          12   0.039   0.045   0.072   0.076  100.0%
                            raw              12   0.299   0.329   0.493   0.551  100.0%
tilt >  45 deg              aligned          88   0.044   0.048   0.073   0.121  100.0%
                            raw              88   0.340   0.367   0.587   0.658  100.0%

worst pose (aligned): tilt 90 deg, error 0.121 mm at nominal position [200. 500. 400.]
